# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 05: Structured Outputs: Context-Free Grammars (CFG) & Constrained Token Sampling

### 1. Conceptual Intuition & Real-World Use Case
When AI agents invoke APIs, databases, or UI components, the output must strictly adhere to a valid JSON schema:
- Prompting an LLM with *"Return valid JSON"* fails at least 5% to 15% of the time (missing brackets, trailing commas, explanatory preamble text).
- Re-prompting on parse failures adds latency and wasted token cost.

Modern LLM inference engines (vLLM, Outlines, Guidance, SGLang) use **Grammar-Constrained Decoding**:
- Converts a JSON schema or Pydantic model into a **Context-Free Grammar (CFG)**.
- At each autoregressive step, the inference engine **masks the logit probabilities** of any token that violates the grammar to $-\infty$.
- **Result: 100% mathematical guarantee of valid JSON structure on the very first forward pass!**

---

### 2. Mathematical Foundations: Logit Masking
Let the vocabulary be $V$, with raw model logits $z \in \mathbb{R}^{|V|}$.
Let the grammar state machine permit a valid token set $V_{\text{valid}} \subset V$ at step $t$:

$$\tilde{z}_i = \begin{cases} z_i & \text{if } i \in V_{\text{valid}} \\ -\infty & \text{if } i \notin V_{\text{valid}} \end{cases}$$

After Softmax:
$$P(\text{token } i) = \frac{\exp(\tilde{z}_i)}{\sum_{j \in V_{\text{valid}}} \exp(\tilde{z}_j)}$$
Invalid tokens have exactly 0.0 probability and can never be sampled!

In [ ]:
import json
import numpy as np

# Step 1: Define Vocabulary & Token IDs
vocab = {
    0: "{",
    1: "}",
    2: '"name"',
    3: ':',
    4: '"Alice"',
    5: '"Bob"',
    6: ',',
    7: '"age"',
    8: '25',
    9: 'invalid_token'
}
token_to_id = {v: k for k, v in vocab.items()}
V_size = len(vocab)

# Desired Target Schema: {"name": "Alice"|"Bob", "age": 25}

# Step 2: Implement Deterministic Grammar State Machine
class JSONGrammarConstraint:
    def __init__(self):
        # States: 0=START, 1=AFTER_OPEN_BRACE, 2=AFTER_NAME_KEY, 3=AFTER_COLON_1,
        #         4=AFTER_NAME_VAL, 5=AFTER_COMMA, 6=AFTER_AGE_KEY, 7=AFTER_COLON_2,
        #         8=AFTER_AGE_VAL, 9=DONE
        self.state = 0

    def get_valid_token_ids(self):
        if self.state == 0:
            return [token_to_id["{"]]
        elif self.state == 1:
            return [token_to_id['"name"']]
        elif self.state == 2:
            return [token_to_id[':']]
        elif self.state == 3:
            return [token_to_id['"Alice"'], token_to_id['"Bob"']]
        elif self.state == 4:
            return [token_to_id[',']]
        elif self.state == 5:
            return [token_to_id['"age"']]
        elif self.state == 6:
            return [token_to_id[':']]
        elif self.state == 7:
            return [token_to_id['25']]
        elif self.state == 8:
            return [token_to_id['}']]
        return []

    def transition(self, token_str):
        if self.state == 0 and token_str == "{":
            self.state = 1
        elif self.state == 1 and token_str == '"name"':
            self.state = 2
        elif self.state == 2 and token_str == ':':
            self.state = 3
        elif self.state == 3 and token_str in ['"Alice"', '"Bob"']:
            self.state = 4
        elif self.state == 4 and token_str == ',':
            self.state = 5
        elif self.state == 5 and token_str == '"age"':
            self.state = 6
        elif self.state == 6 and token_str == ':':
            self.state = 7
        elif self.state == 7 and token_str == '25':
            self.state = 8
        elif self.state == 8 and token_str == '}':
            self.state = 9

# Step 3: Autoregressive Generation with Dynamic Logit Masking
grammar = JSONGrammarConstraint()
generated_tokens = []

np.random.seed(42)

while grammar.state != 9:
    # Simulated raw logits from neural network
    raw_logits = np.random.randn(V_size)
    
    # Get allowed tokens from grammar
    valid_ids = grammar.get_valid_token_ids()
    
    # Apply Logit Masking: set all invalid token logits to -infinity
    masked_logits = np.full(V_size, -np.inf)
    masked_logits[valid_ids] = raw_logits[valid_ids]
    
    # Softmax over valid tokens only
    probs = np.exp(masked_logits - np.max(masked_logits[valid_ids]))
    probs /= np.sum(probs)
    
    # Sample token
    chosen_id = np.random.choice(V_size, p=probs)
    chosen_token = vocab[chosen_id]
    
    generated_tokens.append(chosen_token)
    grammar.transition(chosen_token)

full_output = "".join(generated_tokens)
print("Generated Token Sequence:", generated_tokens)
print("Resulting String:", full_output)

parsed_json = json.loads(full_output)
print("Parsed JSON Object successfully:", parsed_json)
print(f"Extracted Name: {parsed_json['name']}, Age: {parsed_json['age']}")

### 3. Step-by-Step Code Walkthrough
1. **Grammar State Machine:** Tracks the exact parsing phase (expecting key, colon, string, or bracket).
2. **Logit Masking:** Sets invalid tokens to $-\infty$ before softmax normalization.
3. **Guaranteed Compliance:** The output parses cleanly into a Python dictionary with zero syntax errors.

---

### 4. Industry Applications & Trade-offs
- **Tool Calling & Function Execution:** Prevents JSON decoding crashes when models call external APIs.
- **Extraction Pipelines:** Forcing strict schema validation on resumes, invoices, and medical records.

---

### 5. Top Interview Questions & Answers
**Q1: How does grammar-constrained decoding differ from temperature=0 sampling?**
> **Answer:** Temperature=0 (greedy decoding) simply selects the highest logit token. If the model is slightly unaligned, greedy decoding will confidently produce invalid syntax (e.g. starting with ```json). Grammar masking dynamically prunes forbidden transitions, guaranteeing syntactic validity regardless of temperature.

**Q2: What is the performance overhead of grammar-constrained decoding in vLLM / Outlines?**
> **Answer:** Naive token filtering can add latency. Modern engines pre-compile regex or JSON schemas into Deterministic Finite Automata (DFA) index lookups, reducing per-token masking overhead to microseconds, virtually identical to unconstrained generation.